In [ ]:
import sys

API_BASE_URL = 'http://10.198.112.203:10007'
print("Using:", API_BASE_URL)

import httpx
import json
from IPython.display import display, Markdown


class LLMApiClient:
    """Unified client for the LLM API server."""

    def __init__(self, base_url: str, timeout: float = 6000.0):
        self.base_url = base_url.rstrip("/")
        self.token = None
        self.timeout = httpx.Timeout(50.0, read=timeout, write=timeout, pool=timeout)

    def _headers(self):
        return {"Authorization": f"Bearer {self.token}"} if self.token else {}

    # ---- Auth ----

    def login(self, username: str, password: str):
        r = httpx.post(
            f"{self.base_url}/api/auth/login",
            json={"username": username, "password": password},
            timeout=100.0,
        )
        r.raise_for_status()
        self.token = r.json()["access_token"]
        return r.json()

    def list_models(self):
        r = httpx.get(f"{self.base_url}/v1/models", headers=self._headers(), timeout=10.0)
        r.raise_for_status()
        return r.json()

    # ---- Chat ----

    def chat_new(self, model: str, user_message: str):
        """Start a new chat session. Returns (response_text, session_id)."""
        messages = [{"role": "user", "content": user_message}]
        data = {
            "model": model,
            "messages": json.dumps(messages),
        }
        r = httpx.post(
            f"{self.base_url}/v1/chat/completions",
            data=data,
            headers=self._headers(),
            timeout=self.timeout,
        )
        r.raise_for_status()
        result = r.json()
        return result["choices"][0]["message"]["content"], result["x_session_id"]

    def chat_continue(self, model: str, session_id: str, user_message: str):
        """Continue an existing session. Returns (response_text, session_id)."""
        messages = [{"role": "user", "content": user_message}]
        data = {
            "model": model,
            "messages": json.dumps(messages),
            "session_id": session_id,
        }
        r = httpx.post(
            f"{self.base_url}/v1/chat/completions",
            data=data,
            headers=self._headers(),
            timeout=self.timeout,
        )
        r.raise_for_status()
        result = r.json()
        return result["choices"][0]["message"]["content"], result["x_session_id"]

    # ---- RAG management (direct tools-server calls) ----

    def rag_list_collections(self, tools_base: str):
        """List all RAG collections for the authenticated user."""
        r = httpx.get(
            f"{tools_base}/api/tools/rag/collections",
            headers=self._headers(),
            timeout=10.0,
        )
        r.raise_for_status()
        return r.json()

    def rag_list_documents(self, tools_base: str, collection_name: str):
        """List documents in a RAG collection."""
        r = httpx.get(
            f"{tools_base}/api/tools/rag/collections/{collection_name}/documents",
            headers=self._headers(),
            timeout=10.0,
        )
        r.raise_for_status()
        return r.json()

    def rag_query_direct(self, tools_base: str, query: str, collection_name: str, max_results: int = 5):
        """Query RAG directly via the tools server (bypasses the agent)."""
        r = httpx.post(
            f"{tools_base}/api/tools/rag/query",
            headers=self._headers(),
            json={
                "query": query,
                "collection_name": collection_name,
                "max_results": max_results,
            },
            timeout=self.timeout,
        )
        r.raise_for_status()
        return r.json()

    def rag_upload_document(self, tools_base: str, collection_name: str, file_path: str):
        """Upload a local document to a RAG collection."""
        from pathlib import Path
        
        file_path = Path(file_path)
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        
        with open(file_path, "rb") as f:
            files = {"file": (file_path.name, f, "application/octet-stream")}
            data = {"collection_name": collection_name}
            
            r = httpx.post(
                f"{tools_base}/api/tools/rag/upload",
                headers=self._headers(),
                files=files,
                data=data,
                timeout=self.timeout,
            )
        
        r.raise_for_status()
        return r.json()

    def rag_create_collection(self, tools_base: str, collection_name: str):
        """Create a new RAG collection."""
        r = httpx.post(
            f"{tools_base}/api/tools/rag/collections",
            headers=self._headers(),
            json={"collection_name": collection_name},
            timeout=10.0,
        )
        r.raise_for_status()
        return r.json()

    def rag_delete_collection(self, tools_base: str, collection_name: str):
        """Delete a RAG collection."""
        r = httpx.delete(
            f"{tools_base}/api/tools/rag/collections/{collection_name}",
            headers=self._headers(),
            timeout=10.0,
        )
        r.raise_for_status()
        return r.json()

    def rag_delete_document(self, tools_base: str, collection_name: str, document_id: str):
        """Delete a specific document from a collection."""
        r = httpx.delete(
            f"{tools_base}/api/tools/rag/collections/{collection_name}/documents/{document_id}",
            headers=self._headers(),
            timeout=10.0,
        )
        r.raise_for_status()
        return r.json()

# -------------------------------------------------------------------
# Configuration  (adjust to your environment)
# -------------------------------------------------------------------
USERNAME = "admin"
PASSWORD = "administrator"

client = LLMApiClient(API_BASE_URL, timeout=6000.0)
print("✓ Client initialized")
print(f"  Main server : {API_BASE_URL}")

In [ ]:
## Step 1: Authenticate and Discover Model
client.login(USERNAME, PASSWORD)
models = client.list_models()
MODEL = models["data"][0]["id"]

print(f"\u2713 Logged in as: {USERNAME}")
print(f"\u2713 Using model : {MODEL}")
# ## Step 2: Upload Local Documents to RAG (Optional)

# If you want to add your own documents to RAG, use this section. Skip to Step 3 if you already have documents uploaded.

# ### Supported File Formats

# - **Text**: `.txt`, `.md`
# - **Documents**: `.pdf`, `.docx`
# - **Data**: `.json`, `.csv`, `.xlsx`, `.xls`
# # Create a new RAG collection and upload documents
from pathlib import Path

# Step 1: Create collection
collection_name = "default"  # Change this to your desired collection name

try:
    print(f"Creating collection '{collection_name}'...")
    result = client.rag_create_collection(API_BASE_URL, collection_name)
    
    if result.get("success"):
        print(f"✓ Collection created successfully!")
        print(f"  Collection name: {collection_name}\n")
    else:
        print(f"✗ Failed to create collection: {result.get('error')}\n")
except Exception as e:
    print(f"✗ Error creating collection: {e}\n")



In [ ]:
# Step 2: Upload your PDF files
# Replace with your actual PDF file paths
custom_pdf_files = [
    "./USB 3.2 Revision 1.1.pdf",
    "./usb_20.pdf"
]



In [ ]:
print(f"Uploading {len(custom_pdf_files)} documents...\n")

for pdf_file in custom_pdf_files:
    # Check if file exists
    if not Path(pdf_file).exists():
        print(f"⚠️  File not found: {pdf_file}\n")
        continue
    
    print(f"📤 Uploading: {pdf_file}")
    
    try:
        result = client.rag_upload_document(API_BASE_URL, collection_name, pdf_file)
        
        if result.get('success'):
            print(f"  ✓ Success! Chunks created: {result.get('chunks_created')}")
            print(f"  Total chunks in collection: {result.get('total_chunks')}\n")
        else:
            print(f"  ✗ Failed: {result.get('error')}\n")
    except Exception as e:
        print(f"  ✗ Error: {str(e)}\n")

collections_result = client.rag_list_collections(API_BASE_URL)

if collections_result.get("success"):
    collections = collections_result["collections"]
    print(f"Found {len(collections)} collection(s):\n")
    for coll in collections:
        print(f"  Collection : {coll['name']}")
        print(f"  Documents  : {coll['documents']}")
        print(f"  Chunks     : {coll['chunks']}")
        print(f"  Created    : {coll['created_at']}")
        print()
else:
    print("ERROR: Could not list collections.")
    print(collections_result)


def show_latex_result(result):
    """
    Display a dictionary or string result in a LaTeX-formatted way for human readability.
    """
    if isinstance(result, dict):
        latex_str = ""
        # If "answer" or "result" keys, treat them specially
        main_answer = result.get("answer") or result.get("result")
        if main_answer:
            latex_str += r"\textbf{Direct RAG Answer:}\\ " + main_answer.replace("\n", " \\\\ ") + r"\\[1em]"
            # Show sources or metadata if available
            sources = result.get("sources") or result.get("documents") or result.get("docs")
            if sources:
                latex_str += r"\textbf{Sources:}\\ "
                if isinstance(sources, (list, tuple)):
                    latex_str += r"\begin{itemize}"
                    for doc in sources:
                        if isinstance(doc, dict):
                            title = doc.get("title") or doc.get("file_name") or doc.get("id", "Document")
                            c = doc.get("content") or doc.get("excerpt") or ""
                            latex_str += f"\\item \\textbf{{{title}}}: {c[:120].replace('\n', ' ')} ..."
                        else:
                            latex_str += f"\\item {str(doc)[:150]}"
                    latex_str += r"\end{itemize}"
                elif isinstance(sources, str):
                    latex_str += sources
            # Show "score" or other metadata
            score = result.get("score")
            if score is not None:
                latex_str += r"\\[0.5em] \textit{Score:} " + str(score)
        else:
            # Just render whole dict as key-values
            latex_str = r"\begin{tabular}{ll}"
            for k, v in result.items():
                latex_str += f"{k} & {str(v).replace('\n', ' \\\\ ')} \\\\ "
            latex_str += r"\end{tabular}"
    elif isinstance(result, str):
        latex_str = result.replace("\n", " \\\\ ")
    else:
        latex_str = str(result)
    display(Latex(latex_str))

In [ ]:
query = "USB3.2의 LTSSM에 대해서 자세히 설명해주고, 특히 RX.Detect에 대해서 자세히 설명해줘"
direct_result = client.rag_query_direct(
    API_BASE_URL,
    query=query,
    collection_name=collection_name,
    max_results=20,
)

# latex style output of direct_result
# Pretty-print the direct_result in LaTeX style for Jupyter display


show_latex_result(direct_result)


# session_id = compare_responses(query_4, collection_name, MODEL, use_markdown=True)